# Inter-basis operators

Once `DoubleBasis` is clear as map metadata, we can use it to build non-square operators. This notebook shows how `operator(...)` and `trans_inv_operator(...)` work when the domain and codomain are different bases.

The main thing to watch is the shape: rows belong to the target basis, columns belong to the source basis.

In [ ]:
import Pkg
using LinearAlgebra
using SparseArrays

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
    include(joinpath(REPO_ROOT, "src", "EDKit.jl"))
    using .EDKit
else
    using EDKit
end


## A rectangular one-site operator

We again map from the full Hilbert space into a symmetry sector. This time we build a one-site `Z` operator with `DoubleBasis(Btarget, Bsource)`. The result is a rectangular matrix with one row per target basis state and one column per source basis state.

In [ ]:
L = 4
Bsource = TensorBasis(L = L, base = 2)
Btarget = basis(L = L, N = 2, k = 0, p = 1, base = 2, threaded = false)
T = DoubleBasis(Btarget, Bsource)

Z1 = operator(spin("Z"), [1], T)
Z1_dense = Array(Z1)
Z1_sparse = sparse(Z1)

summary_one_site = (
    operator_size = size(Z1_dense),
    nnz_sparse = nnz(Z1_sparse),
    source_dimension = size(Bsource, 1),
    target_dimension = size(Btarget, 1),
)
summary_one_site


## Apply the rectangular operator

A vector in the source basis can be acted on directly. The output is a vector in target-basis coordinates. This is often a more useful mental model than thinking about the dense rectangular matrix itself.

In [ ]:
psi_full = randn(ComplexF64, size(Bsource, 1)) |> normalize
psi_target = Z1 * psi_full
psi_target_dense = Z1_dense * psi_full

summary_apply = (
    output_length = length(psi_target),
    linear_vs_dense_error = norm(psi_target - psi_target_dense),
)
summary_apply


## Translation-invariant two-site operator

The same idea works for translation-invariant operator construction. Here we build a nearest-neighbour XXZ term and ask for its rectangular representation from the full basis into the symmetry sector.

In [ ]:
h2 = spin((1.0, "xx"), (1.0, "yy"), (0.7, "zz"))
Hrect = trans_inv_operator(h2, 1:2, T)
Hrect_dense = Array(Hrect)

summary_trans_inv = (
    hamiltonian_size = size(Hrect_dense),
    frobenius_norm = norm(Hrect_dense),
)
summary_trans_inv


## Compare with explicit symmetrization

There is a simple consistency check. If `Hfull` is the full-space Hamiltonian and `P = symmetrizer(T)`, then the rectangular operator should match `P * Hfull` when the source basis is full and the target basis is the symmetry sector.

In [ ]:
Hfull = trans_inv_operator(h2, 1:2, Bsource) |> Array
P = symmetrizer(T)
Hprojected = P * Hfull

summary_compare = (
    rectangular_consistency_error = norm(Hrect_dense - Hprojected),
)
summary_compare


## Key takeaway

With `DoubleBasis`, non-square operators are not a special case bolted onto EDKit. They are ordinary `Operator` objects whose domain and codomain happen to be different bases. That makes it straightforward to move between full-space calculations and symmetry-resolved calculations without rewriting the local operator data.